# Gemma + LoRA Setup (LLaMA-Factory, 2xGPU Ready)

This notebook assumes you are using your `ml-env` Python environment and have `LLaMA-Factory` available at `./LLaMA-Factory`.

It does three things:
1. Loads a Gemma model with `transformers`.
2. Wraps it with PEFT LoRA adapters.
3. Generates a ready-to-run LLaMA-Factory LoRA training YAML.



In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/home/rameyjm7/workspace/llamafactory-gemma-lora')
LLAMA_FACTORY_DIR = PROJECT_ROOT / 'LLaMA-Factory'
CONFIG_PATH = PROJECT_ROOT / 'gemma2_lora_sft.yaml'

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'LLAMA_FACTORY_DIR exists: {LLAMA_FACTORY_DIR.exists()}')
print(f'Config will be written to: {CONFIG_PATH}')


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_NAME_OR_PATH = 'google/gemma-2-9b-it'
TEMPLATE_NAME = 'gemma2'  # LLaMA-Factory template for Gemma 2

if torch.cuda.is_available():
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    DEVICE_MAP = 'auto'
else:
    DTYPE = torch.float32
    DEVICE_MAP = None

print('Model:', MODEL_NAME_OR_PATH)
print('Template:', TEMPLATE_NAME)
print('Dtype:', DTYPE)
print('Device map:', DEVICE_MAP)


In [ ]:
import os
import subprocess
from pathlib import Path
import pickle

ENV_FILE = Path('/home/rameyjm7/workspace/llamafactory-gemma-lora/env.txt')
ENV_PKL_FILE = Path('/home/rameyjm7/workspace/llamafactory-gemma-lora/configs/env.pkl')
MODEL_REPO = 'google/gemma-2-9b-it'
LOCAL_MODEL_DIR = Path('/scratch/rameyjm7/models/google__gemma-2-9b-it')
LOCAL_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)


def load_hf_token(env_path: Path, pkl_path: Path) -> str:
    if pkl_path.exists():
        try:
            with pkl_path.open('rb') as f:
                payload = pickle.load(f)
            token = str(payload.get('HF_TOKEN', '')).strip()
            if token:
                return token
        except Exception as exc:
            print(f'Warning: failed to read {pkl_path}: {exc}')

    if not env_path.exists():
        raise FileNotFoundError(f'Env file not found: {env_path}')

    token = ''
    for raw in env_path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('HF_TOKEN='):
            token = line.split('=', 1)[1].strip().strip('\"').strip("'")
            break

    if not token:
        raise ValueError(f'HF_TOKEN not found or empty in {env_path} or {pkl_path}')

    return token

hf_token = load_hf_token(ENV_FILE, ENV_PKL_FILE)
os.environ['HF_TOKEN'] = hf_token

# Avoid very short hub timeouts on large files.
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '60')
os.environ.setdefault('HF_HUB_DOWNLOAD_TIMEOUT', '1800')

cmd = [
    'huggingface-cli', 'download', MODEL_REPO,
    '--token', hf_token,
    '--local-dir', str(LOCAL_MODEL_DIR),
    '--local-dir-use-symlinks', 'False',
]

print(f'Loaded HF_TOKEN using {ENV_PKL_FILE} with fallback to {ENV_FILE}')
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

print('Model downloaded to:', LOCAL_MODEL_DIR)
MODEL_NAME_OR_PATH = str(LOCAL_MODEL_DIR)
print('MODEL_NAME_OR_PATH set to local path:', MODEL_NAME_OR_PATH)





In [ ]:
# Optional smoke test only. Keep disabled unless you explicitly want to load the full base model now.
RUN_MODEL_SMOKE_TEST = False

if RUN_MODEL_SMOKE_TEST:
    token = os.environ.get('HF_TOKEN')
    if not token:
        raise ValueError('HF_TOKEN is missing. Run the auth cell above first.')

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME_OR_PATH,
        token=token,
        cache_dir='/home/rameyjm7/workspace/llamafactory-gemma-lora/outputs/hf_cache/hf_cache',
        use_fast=False,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME_OR_PATH,
        torch_dtype=DTYPE,
        device_map=DEVICE_MAP,
        token=token,
        cache_dir='/home/rameyjm7/workspace/llamafactory-gemma-lora/outputs/hf_cache/hf_cache',
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print('Loaded tokenizer + model successfully.')
else:
    print('Skipping base model smoke test (RUN_MODEL_SMOKE_TEST=False).')



In [ ]:
if 'model' in globals():
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
else:
    print('Model not loaded; skipping in-memory PEFT wrap demo.')


In [ ]:
import textwrap

precision_line = 'bf16: true' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'fp16: true'

yaml_text = textwrap.dedent(f'''
### model
model_name_or_path: {MODEL_NAME_OR_PATH}
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all

### dataset
dataset: identity,alpaca_en_demo
template: {TEMPLATE_NAME}
cutoff_len: 2048
max_samples: 1000
preprocessing_num_workers: 8
dataloader_num_workers: 2

### output
output_dir: saves/gemma-2-9b-it/lora/sft
logging_steps: 10
save_steps: 200
plot_loss: true
overwrite_output_dir: true
save_only_model: false
report_to: none

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
{precision_line}
ddp_timeout: 180000000
resume_from_checkpoint: null
''').strip() + '\n'

CONFIG_PATH.write_text(yaml_text, encoding='utf-8')
print(f'Wrote config: {CONFIG_PATH}')
print(yaml_text)


## Start LoRA Training with LLaMA-Factory

Run the next cell when ready.



In [ ]:
print('Legacy demo train cell disabled.')
print('Use the EXPRESS section below with RUN_TRAIN / RUN_INFERENCE toggles.')


## EXPRESS LoRA + Figure-2 Comparison (Notebook-only)

This section keeps the full workflow inside the notebook:
1. Build an EXPRESS SFT dataset for LLaMA-Factory.
2. Train Gemma LoRA.
3. Run inference with the LoRA adapter.
4. Evaluate metrics.
5. Plot your LoRA result against Figure-2 baseline points.



In [ ]:
import os
import re
import ast
import json
import textwrap
import subprocess
import shlex
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('/home/rameyjm7/workspace/llamafactory-gemma-lora')
EXPRESS_ROOT = PROJECT_ROOT / 'express-emotion-recognition'
LF_ROOT = PROJECT_ROOT / 'LLaMA-Factory'

LORA_DATA_DIR = PROJECT_ROOT / 'lora_data'
LORA_RUN_DIR = PROJECT_ROOT / 'outputs' / 'lora_runs' / 'gemma2_9b_express_lora'
LORA_CONFIG_PATH = PROJECT_ROOT / 'gemma2_9b_express_lora.yaml'
HF_CACHE_DIR = PROJECT_ROOT / 'outputs' / 'hf_cache' / 'hf_cache'

LORA_DATA_DIR.mkdir(parents=True, exist_ok=True)
LORA_RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('EXPRESS_ROOT exists:', EXPRESS_ROOT.exists())
print('LF_ROOT exists:', LF_ROOT.exists())



In [ ]:
def run_cmd(cmd, cwd=None, env=None, check=True):
    printable = ' '.join(shlex.quote(str(x)) for x in cmd)
    print(f'$ {printable}')
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)

    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        lines.append(line)

    rc = proc.wait()
    out = ''.join(lines)
    if check and rc != 0:
        tail = ''.join(lines[-80:])
        raise RuntimeError(
            f'Command failed ({rc}): {printable}\n'
            f'--- Last output lines ---\n{tail}'
        )
    return out


def parse_eval_metrics(stdout_text):
    metrics = {}
    for key in ['VRate', 'AccL', 'AccV', 'F1V', 'AccV-2']:
        m = re.search(rf'{re.escape(key)}\s*=\s*([0-9]*\.?[0-9]+)', stdout_text)
        if m:
            metrics[key] = float(m.group(1))
    return metrics


def resolve_adapter_path(run_dir: Path) -> Path:
    # Prefer the best checkpoint selected during training when available.
    trainer_state = run_dir / 'trainer_state.json'
    if trainer_state.exists():
        try:
            state = json.loads(trainer_state.read_text(encoding='utf-8'))
            best_ckpt = state.get('best_model_checkpoint')
            if best_ckpt:
                best_path = Path(best_ckpt)
                if not best_path.is_absolute():
                    best_path = run_dir / best_path
                if (best_path / 'adapter_config.json').exists():
                    print(f'Using best checkpoint from trainer_state.json: {best_path}')
                    return best_path
        except Exception as exc:
            print(f'Warning: could not parse trainer_state.json for best checkpoint: {exc}')

    if (run_dir / 'adapter_config.json').exists():
        return run_dir

    checkpoints = sorted(run_dir.glob('checkpoint-*'))
    checkpoints = [p for p in checkpoints if (p / 'adapter_config.json').exists()]
    if checkpoints:
        return checkpoints[-1]

    raise FileNotFoundError(
        f'Could not find adapter_config.json in {run_dir} or any checkpoint-* folder.'
    )




In [ ]:
import pickle
# Cell 0: Configure toggles, token, and model path.
RUN_TRAIN = True       # Set True to launch LoRA training.
RUN_INFERENCE = True   # Set True to run LoRA inference + evaluation.

# Multi-GPU training controls for LLaMA-Factory (DDP via torchrun).
TRAIN_GPU_IDS = '0,1'
TRAIN_NPROC_PER_NODE = 2
TRAIN_MASTER_PORT = 29651

ENV_FILE = Path('/home/rameyjm7/workspace/llamafactory-gemma-lora/env.txt')
ENV_PKL_FILE = Path('/home/rameyjm7/workspace/llamafactory-gemma-lora/configs/env.pkl')

def load_hf_token(env_path: Path, pkl_path: Path) -> str:
    if pkl_path.exists():
        try:
            with pkl_path.open('rb') as f:
                payload = pickle.load(f)
            token = str(payload.get('HF_TOKEN', '')).strip()
            if token:
                return token
        except Exception as exc:
            print(f'Warning: failed to read {pkl_path}: {exc}')

    if not env_path.exists():
        raise FileNotFoundError(f'Env file not found: {env_path}')

    token = ''
    for raw in env_path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('HF_TOKEN='):
            token = line.split('=', 1)[1].strip().strip('\"').strip("'")
            break

    if not token:
        raise ValueError(f'HF_TOKEN not found or empty in {env_path} or {pkl_path}')

    return token

HF_TOKEN = load_hf_token(ENV_FILE, ENV_PKL_FILE)
os.environ['HF_TOKEN'] = HF_TOKEN

LOCAL_MODEL_DIR = Path('/scratch/rameyjm7/models/google__gemma-2-9b-it')
if not LOCAL_MODEL_DIR.exists():
    raise FileNotFoundError(
        f'Local model path not found: {LOCAL_MODEL_DIR}. Download the model to /scratch first.'
    )
MODEL_NAME = str(LOCAL_MODEL_DIR)
TEMPLATE_NAME = 'gemma2'

TRAIN_ROWS = 12000      # use None for full express.csv
TRAIN_PER_DEVICE_BATCH_SIZE = 8
TRAIN_GRAD_ACCUM_STEPS = 2
BATCH_SIZE_INFER = 8
DISABLE_8BIT_INFER = True

assert EXPRESS_ROOT.exists(), 'Missing express-emotion-recognition repo.'
assert LF_ROOT.exists(), 'Missing LLaMA-Factory repo.'


# Resolve an effective launch config to avoid duplicate-GPU NCCL failures.
requested_gpu_ids = [x.strip() for x in str(TRAIN_GPU_IDS).split(',') if x.strip()]
if not requested_gpu_ids:
    requested_gpu_ids = ['0']

try:
    import torch
    detected_gpus = int(torch.cuda.device_count())
except Exception:
    detected_gpus = len(requested_gpu_ids)

unique_requested = len(set(requested_gpu_ids))
if detected_gpus >= 2 and unique_requested >= 2 and int(TRAIN_NPROC_PER_NODE) >= 2:
    EFFECTIVE_GPU_IDS = ','.join(requested_gpu_ids[:int(TRAIN_NPROC_PER_NODE)])
    EFFECTIVE_NPROC = int(TRAIN_NPROC_PER_NODE)
    TRAIN_USE_TORCHRUN = True
else:
    EFFECTIVE_GPU_IDS = requested_gpu_ids[0]
    EFFECTIVE_NPROC = 1
    TRAIN_USE_TORCHRUN = False
    print(
        f'Warning: forcing single-GPU launch (detected_gpus={detected_gpus}, '
        f'requested_unique={unique_requested}, requested_nproc={TRAIN_NPROC_PER_NODE}).'
    )

print('RUN_TRAIN =', RUN_TRAIN)
print('RUN_INFERENCE =', RUN_INFERENCE)
print('MODEL_NAME =', MODEL_NAME)
print('TRAIN_ROWS =', TRAIN_ROWS)
print('TRAIN_GPU_IDS (requested) =', TRAIN_GPU_IDS)
print('EFFECTIVE_GPU_IDS =', EFFECTIVE_GPU_IDS)
print('TRAIN_NPROC_PER_NODE (requested) =', TRAIN_NPROC_PER_NODE)
print('EFFECTIVE_NPROC =', EFFECTIVE_NPROC)
print('TRAIN_USE_TORCHRUN =', TRAIN_USE_TORCHRUN)
print('TRAIN_PER_DEVICE_BATCH_SIZE =', TRAIN_PER_DEVICE_BATCH_SIZE)
print('TRAIN_GRAD_ACCUM_STEPS =', TRAIN_GRAD_ACCUM_STEPS)
print(f'Loaded HF_TOKEN using {ENV_PKL_FILE} with fallback to {ENV_FILE}')





In [ ]:
# Cell 1: Build a local Alpaca-format training dataset for EXPRESS.
express_path = EXPRESS_ROOT / 'data' / 'express.csv'
train_json_path = LORA_DATA_DIR / 'express_emotion_lora_train.json'
dataset_info_path = LORA_DATA_DIR / 'dataset_info.json'

src_df = pd.read_csv(express_path)
if TRAIN_ROWS is not None:
    src_df = src_df.sample(n=min(TRAIN_ROWS, len(src_df)), random_state=42).reset_index(drop=True)

records = []
for _, row in src_df.iterrows():
    labels_raw = row['labels']
    try:
        labels = ast.literal_eval(labels_raw) if isinstance(labels_raw, str) else labels_raw
    except Exception:
        continue
    if not isinstance(labels, list) or len(labels) == 0:
        continue

    labels = [str(x).strip().lower() for x in labels]
    n_labels = int(row['number_of_labels']) if 'number_of_labels' in row and pd.notna(row['number_of_labels']) else len(labels)

    instruction = (
        f"Predict the {n_labels} masked emotion word(s) from the text. "
        f"Return only a Python list of exactly {n_labels} lowercase emotion words."
    )

    records.append({
        'instruction': instruction,
        'input': str(row['segment']),
        'output': json.dumps(labels),
    })

with open(train_json_path, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

dataset_info = {
    'express_emotion_lora_train': {
        'file_name': train_json_path.name,
        'columns': {
            'prompt': 'instruction',
            'query': 'input',
            'response': 'output',
        },
    }
}
with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, indent=2)

print('Wrote training records:', len(records))
print('Training file:', train_json_path)
print('Dataset info file:', dataset_info_path)


In [ ]:
# Cell 2: Write LoRA train config (local dataset_dir).
precision_line = 'bf16: true' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'fp16: true'

yaml_text = textwrap.dedent(f'''
### model
model_name_or_path: {MODEL_NAME}
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all
early_stopping_steps: 5

### dataset
dataset_dir: {LORA_DATA_DIR}
dataset: express_emotion_lora_train
val_size: 0.05
template: {TEMPLATE_NAME}
cutoff_len: 1024
preprocessing_num_workers: 8
dataloader_num_workers: 2

### output
output_dir: {LORA_RUN_DIR}
logging_steps: 20
save_strategy: steps
save_steps: 100
eval_strategy: steps
eval_steps: 100
load_best_model_at_end: true
metric_for_best_model: eval_loss
greater_is_better: false
save_total_limit: 3
plot_loss: true
overwrite_output_dir: true
save_only_model: false
report_to: none

### train
per_device_train_batch_size: {TRAIN_PER_DEVICE_BATCH_SIZE}
gradient_accumulation_steps: {TRAIN_GRAD_ACCUM_STEPS}
per_device_eval_batch_size: 2
learning_rate: 1.0e-4
num_train_epochs: 2.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
{precision_line}
ddp_timeout: 180000000
resume_from_checkpoint: null
''').strip() + '\n'

LORA_CONFIG_PATH.write_text(yaml_text, encoding='utf-8')
print('Wrote config:', LORA_CONFIG_PATH)
print(yaml_text)




In [ ]:
# Cell 3: Train LoRA from notebook.
if RUN_TRAIN:
    _ = run_cmd(
        ['llamafactory-cli', 'train', str(LORA_CONFIG_PATH)],
        cwd=LF_ROOT,
        env={
            'HF_TOKEN': HF_TOKEN,
            'CUDA_VISIBLE_DEVICES': EFFECTIVE_GPU_IDS,
            'NPROC_PER_NODE': str(EFFECTIVE_NPROC),
            'FORCE_TORCHRUN': '1' if TRAIN_USE_TORCHRUN else '0',
            'MASTER_PORT': str(TRAIN_MASTER_PORT),
        },
        check=True,
    )
else:
    print('Training skipped. Set RUN_TRAIN=True to run it.')



In [ ]:
# Cell 4: Build eval input aligned to baseline rows.
baseline_result_path = EXPRESS_ROOT / 'results' / 'gemma2_9B_it.csv'
lora_eval_input = EXPRESS_ROOT / 'baseline_eval_input_for_lora.csv'
lora_raw_output = EXPRESS_ROOT / 'baseline_raw_gemma2_9b_lora.csv'
lora_clean_output = EXPRESS_ROOT / 'baseline_clean_gemma2_9b_lora.csv'

base_df = pd.read_csv(baseline_result_path)
cols_needed = [
    'index', 'original_index', 'original_ids', 'segment',
    'number_of_labels', 'labels', 'original_labels', 'word_count'
]
base_eval_df = base_df[cols_needed].copy()
base_eval_df['output'] = None
base_eval_df.to_csv(lora_eval_input, index=False)

print('Prepared eval input rows:', len(base_eval_df))
print('Eval input:', lora_eval_input)


In [ ]:
# Cell 5: Run LoRA inference, cleaning, and evaluation.
if RUN_INFERENCE:
    adapter_path = resolve_adapter_path(LORA_RUN_DIR)
    print('Using adapter path:', adapter_path)


    # Preflight cleanup: remove stale empty/truncated files that break resume logic in model-inference.py.
    for stale_path in [lora_raw_output, lora_clean_output]:
        if stale_path.exists():
            try:
                if stale_path.stat().st_size == 0:
                    stale_path.unlink()
                    print('Removed empty stale file:', stale_path)
            except Exception as exc:
                print(f'Warning: could not inspect/remove {stale_path}: {exc}')

    infer_cmd = [
        'python', str(EXPRESS_ROOT / 'src' / 'model-inference.py'),
        '--model', MODEL_NAME,
        '--adapter_name_or_path', str(adapter_path),
        '--input', str(lora_eval_input),
        '--output', str(lora_raw_output),
        '--download_path', str(HF_CACHE_DIR),
        '--batch_size', str(BATCH_SIZE_INFER),
    ]
    if DISABLE_8BIT_INFER:
        infer_cmd.append('--disable_8bit')

    _ = run_cmd(infer_cmd, cwd=PROJECT_ROOT, env={'HF_TOKEN': HF_TOKEN}, check=True)


    # Guardrail: normalize raw inference outputs so result-cleaning.py doesn't crash
    # when parsed lists contain non-string entries (e.g., ints/floats).
    try:
        import ast
        import pandas as pd

        def _to_list_of_strings(val):
            # Returns either list[str] or original scalar/string.
            if isinstance(val, list):
                return [str(x).strip() for x in val]
            if isinstance(val, str):
                s = val.strip()
                if not s:
                    return s
                if s.startswith('[') and s.endswith(']'):
                    try:
                        parsed = ast.literal_eval(s)
                    except Exception:
                        # Keep as-is; downstream cleaner will handle invalid outputs.
                        return s
                    if isinstance(parsed, list):
                        return [str(x).strip() for x in parsed]
                return s
            return val

        if lora_raw_output.exists() and lora_raw_output.stat().st_size > 0:
            _raw_df = pd.read_csv(lora_raw_output)

            # Normalize the exact source column used by result-cleaning.py
            if 'output' in _raw_df.columns:
                _raw_df['output'] = _raw_df['output'].apply(_to_list_of_strings)
                # Serialize lists back as Python-list literals with quoted strings.
                _raw_df['output'] = _raw_df['output'].apply(lambda v: repr(v) if isinstance(v, list) else v)

            # Normalize labels too for symmetry/safety.
            if 'labels' in _raw_df.columns:
                _raw_df['labels'] = _raw_df['labels'].apply(_to_list_of_strings)
                _raw_df['labels'] = _raw_df['labels'].apply(lambda v: repr(v) if isinstance(v, list) else v)

            # Optional compatibility column if present in raw file.
            if 'output_formatted' in _raw_df.columns:
                _raw_df['output_formatted'] = _raw_df['output_formatted'].apply(_to_list_of_strings)
                _raw_df['output_formatted'] = _raw_df['output_formatted'].apply(lambda v: repr(v) if isinstance(v, list) else v)

            _raw_df.to_csv(lora_raw_output, index=False)
            print('Normalized raw inference CSV for cleaning:', lora_raw_output)
    except Exception as exc:
        print('Warning: raw output normalization skipped:', exc)

    _ = run_cmd([
        'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-cleaning.py'),
        '--input', str(lora_raw_output),
        '--output', str(lora_clean_output),
    ], cwd=PROJECT_ROOT, check=True)

    eval_stdout = run_cmd([
        'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-evaluation.py'),
        '--result_path', str(lora_clean_output),
        '--breakdowns_path', str(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv'),
    ], cwd=PROJECT_ROOT, check=True)

    lora_metrics = parse_eval_metrics(eval_stdout)
    print('LoRA metrics:', lora_metrics)
else:
    print('Inference/eval skipped. Set RUN_INFERENCE=True to run it.')


In [ ]:
from datetime import datetime, timezone
# Cell 6A: Upsert this run into shared metrics table for notebook 60.
METRICS_TABLE_PATH = PROJECT_ROOT / 'outputs' / 'metrics' / 'all_models_metrics.csv'

row = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
    'model_id': 'gemma2_9b_it',
    'family': 'gemma2',
    'model_name_or_path': MODEL_NAME,
    'size_b': 9.0,
    'pipeline': 'llamafactory_lora_causal',
    'is_lora': True,
    'v_rate': lora_metrics.get('VRate') if 'lora_metrics' in globals() else None,
    'accl': lora_metrics.get('AccL') if 'lora_metrics' in globals() else None,
    'accv': lora_metrics.get('AccV') if 'lora_metrics' in globals() else None,
    'f1v': lora_metrics.get('F1V') if 'lora_metrics' in globals() else None,
    'accv2': lora_metrics.get('AccV-2') if 'lora_metrics' in globals() else None,
    'clean_output_path': str(lora_clean_output) if 'lora_clean_output' in globals() else None,
    'run_dir': str(LORA_RUN_DIR),
}

if METRICS_TABLE_PATH.exists():
    metrics_df = pd.read_csv(METRICS_TABLE_PATH)
else:
    metrics_df = pd.DataFrame()

if not metrics_df.empty and {'model_id', 'is_lora'}.issubset(metrics_df.columns):
    metrics_df = metrics_df[~((metrics_df['model_id'] == row['model_id']) & (metrics_df['is_lora'].astype(str).str.lower() == 'true'))]

metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)
metrics_df.to_csv(METRICS_TABLE_PATH, index=False)

print('Updated metrics table:', METRICS_TABLE_PATH)
display(metrics_df.tail(12))



In [ ]:
# Cell 7: Compare your LoRA result against the Figure-2 baseline chart.
# This recomputes baseline AccV from results/*.csv so the comparison is reproducible.
lex_df = pd.read_csv(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv')
vec_map = {str(r['word']).lower(): tuple(int(v) for v in r.iloc[1:].tolist()) for _, r in lex_df.iterrows()}
zero_vec = tuple([0] * 10)

_parse_cache = {}
def parse_list_cached(raw):
    if isinstance(raw, list):
        return [str(x).strip().lower() for x in raw]
    if not isinstance(raw, str):
        return None
    s = raw.strip()
    if s == 'INVALID OUTPUT':
        return None
    if s in _parse_cache:
        return _parse_cache[s]
    try:
        v = ast.literal_eval(s)
        parsed = [str(x).strip().lower() for x in v] if isinstance(v, list) else None
    except Exception:
        parsed = None
    _parse_cache[s] = parsed
    return parsed


def compute_accv(csv_path):
    df = pd.read_csv(csv_path, usecols=['labels', 'output_formatted', 'number_of_labels'])
    total_masks = int(df['number_of_labels'].sum())
    matches = 0
    for raw_labels, raw_preds in zip(df['labels'].tolist(), df['output_formatted'].tolist()):
        labels = parse_list_cached(raw_labels)
        preds = parse_list_cached(raw_preds)
        if labels is None or preds is None:
            continue
        for t, p in zip(labels, preds):
            matches += int(vec_map.get(t, zero_vec) == vec_map.get(p, zero_vec))
    return (matches / total_masks) if total_masks > 0 else float('nan')

baseline_models = [
    {'label': 'Flan-T5-large', 'family': 'Flan-T5', 'size_b': 0.78, 'file': 'flan_t5_large.csv'},
    {'label': 'Flan-T5-xl', 'family': 'Flan-T5', 'size_b': 3.0, 'file': 'flan_t5_xl.csv'},
    {'label': 'Flan-T5-xxl', 'family': 'Flan-T5', 'size_b': 11.0, 'file': 'flan_t5_xxl.csv'},
    {'label': 'Gpt-3.5-turbo', 'family': 'GPT', 'size_b': 175.0, 'file': 'gpt_35_turbo_0125.csv'},
    {'label': 'Gpt-4o', 'family': 'GPT', 'size_b': 1750.0, 'file': 'gpt_4o.csv'},
    {'label': 'Gemma-2-2B-it', 'family': 'Gemma2', 'size_b': 2.0, 'file': 'gemma2_2B_it.csv'},
    {'label': 'Gemma-2-9B-it', 'family': 'Gemma2', 'size_b': 9.0, 'file': 'gemma2_9B_it.csv'},
    {'label': 'Gemma-2-27B-it', 'family': 'Gemma2', 'size_b': 27.0, 'file': 'gemma2_27B_it.csv'},
    {'label': 'Llama3.1-8B-instruct', 'family': 'Llama3.1', 'size_b': 8.0, 'file': 'llama3.1_8B_instruct.csv'},
    {'label': 'Llama3.1-70B-instruct', 'family': 'Llama3.1', 'size_b': 70.0, 'file': 'llama3.1_70B_instruct.csv'},
    {'label': 'Longformer-base', 'family': 'Longformer', 'size_b': 0.149, 'file': 'longformer.csv'},
    {'label': 'Mental-Longformer-base', 'family': 'Longformer', 'size_b': 0.149, 'file': 'mental-longformer.csv'},
    {'label': 'Roberta-base', 'family': 'Roberta', 'size_b': 0.125, 'file': 'roberta-base.csv'},
    {'label': 'Mental-Roberta-base', 'family': 'Roberta', 'size_b': 0.125, 'file': 'mental-roberta-base.csv'},
]

rows = []
for m in baseline_models:
    p = EXPRESS_ROOT / 'results' / m['file']
    rows.append({**m, 'accv': compute_accv(p)})

fig_df = pd.DataFrame(rows)

lora_accv = None
if 'lora_metrics' in globals() and 'AccV' in lora_metrics:
    lora_accv = float(lora_metrics['AccV'])
elif lora_clean_output.exists():
    lora_accv = compute_accv(lora_clean_output)

sns.set_theme(style='whitegrid')
family_colors = {
    'Flan-T5': '#4C72B0',
    'GPT': '#DD8452',
    'Gemma2': '#55A868',
    'Llama3.1': '#C44E52',
    'Longformer': '#8172B2',
    'Roberta': '#937860',
}
family_markers = {
    'Flan-T5': 'D',
    'GPT': 'P',
    'Gemma2': '^',
    'Llama3.1': 'X',
    'Longformer': 's',
    'Roberta': 'o',
}

plt.figure(figsize=(14, 6))
for fam in ['Flan-T5', 'GPT', 'Gemma2', 'Llama3.1', 'Longformer', 'Roberta']:
    sub = fig_df[fig_df['family'] == fam].sort_values('size_b')
    if sub.empty:
        continue
    plt.plot(
        sub['size_b'], sub['accv'],
        marker=family_markers[fam],
        color=family_colors[fam],
        linestyle='--',
        linewidth=1.25,
        markersize=8,
        label=fam,
    )

if lora_accv is not None:
    plt.scatter([9.0], [lora_accv], marker='*', s=320, color='black', label='Gemma-2-9B-it + LoRA')
    plt.text(9.0 * 1.03, lora_accv + 0.004, 'Gemma-2-9B-it + LoRA', fontsize=10)
    print('LoRA AccV:', lora_accv)
else:
    print('No LoRA metric found yet. Run inference/evaluation first.')

plt.xscale('log')
plt.xlabel('Model Size')
plt.ylabel('Accuracy (Vector)')
plt.ylim(0.08, 0.40)
plt.xticks([0.1, 1, 10, 100, 1000], ['100M', '1B', '10B', '100B', '1000B'])
plt.title('Figure 2 Baseline + Your Gemma LoRA Result')
plt.legend(loc='lower left')
plt.tight_layout()

overlay_path = PROJECT_ROOT / 'figure2_with_lora_overlay.png'
plt.savefig(overlay_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved overlay chart:', overlay_path)

